# 🏭 Mustererkennung in einem Produktionsprozess
## Wälzlager-Fehlerdiagnose mit dem CWRU Bearing Dataset

---

### Überblick

In dieser interaktiven Demo untersuchen wir, wie **Mustererkennung** eingesetzt werden kann,
um Fehler in Wälzlagern automatisch zu erkennen — ein zentrales Thema der **vorausschauenden Wartung (Predictive Maintenance)**.

**Warum ist das wichtig?**
- Wälzlager sind in fast jeder rotierenden Maschine verbaut (Motoren, Pumpen, Turbinen)
- ~40-50% aller Maschinenausfälle gehen auf Lagerdefekte zurück
- Ungeplante Stillstände kosten die Industrie Milliarden pro Jahr
- Frühzeitige Fehlererkennung durch Vibrationssignale kann Ausfälle verhindern

### Was wir heute tun:

| Schritt | Beschreibung |
|---------|-------------|
| **1. Datenexploration** | Vibrationssignale visualisieren und verstehen |
| **2. Feature-Extraktion** | Handgemachte Merkmale aus den Signalen berechnen |
| **3. Klassisches ML** | Random Forest auf extrahierten Features trainieren |
| **4. Deep Learning** | 1D-CNN direkt auf Rohsignalen trainieren |
| **5. Vergleich** | Beide Ansätze gegenüberstellen |

### Dataset: CWRU Bearing Data Center

- **Quelle:** Case Western Reserve University
- **Sensoren:** Beschleunigungssensoren am Lagergehäuse (12 kHz Abtastrate)
- **Fehlertypen:** Normal, Innenring (IR), Außenring (OR), Kugel (B)
- **Fehlerdurchmesser:** 0.007, 0.014, 0.021 Zoll (künstlich eingebrachte Defekte)

---

## 0. Setup & Bibliotheken laden

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Projektverzeichnis zum Pfad hinzufügen
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Eigene Module importieren
from src.data_loader import (
    load_mat_files, segment_signals, normalize_segments,
    prepare_dataset, generate_demo_data, CLASS_NAMES, LABEL_NAMES
)
from src.features import (
    extract_all_features, extract_features_single,
    FEATURE_NAMES, _compute_spectrum
)
from src.classical_model import train_classical, evaluate_classical
from src.cnn_model import build_cnn, train_cnn, evaluate_cnn, get_model_summary

# Plot-Einstellungen
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Farben für die 4 Klassen
COLORS = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']

print('✓ Alle Bibliotheken geladen!')
print(f'  NumPy:       {np.__version__}')
print(f'  Pandas:      {pd.__version__}')
print(f'  Matplotlib:  {plt.matplotlib.__version__}')

## 1. Daten laden

Wir versuchen zuerst, den echten CWRU-Dataset aus dem `data/`-Ordner zu laden.
Falls dieser noch nicht heruntergeladen wurde, verwenden wir **synthetische Demo-Daten**,
die das Verhalten echter Wälzlager-Signale simulieren.

> **💡 Hinweis:** Den echten Dataset können Sie hier herunterladen:
> https://www.kaggle.com/datasets/brjapon/cwru-bearing-datasets/data
> 
> Entpacken Sie die `.mat`-Dateien in den Ordner `data/`.

In [ ]:
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
USE_DEMO_DATA = False

# Versuche echte Daten zu laden
try:
    mat_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.mat')]
    if len(mat_files) == 0:
        raise FileNotFoundError('Keine .mat-Dateien gefunden')
    
    print(f'📂 {len(mat_files)} .mat-Dateien gefunden in {DATA_DIR}')
    data = prepare_dataset(DATA_DIR, segment_length=1024, overlap=0.5)
    records = data['records']
    
except (FileNotFoundError, ValueError) as e:
    print(f'⚠ Echte Daten nicht verfügbar: {e}')
    print('→ Verwende synthetische Demo-Daten\n')
    USE_DEMO_DATA = True
    data = generate_demo_data(n_per_class=500, segment_length=1024)
    records = None

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

print(f'\n📊 Datensatz-Zusammenfassung:')
print(f'  Trainings-Segmente: {X_train.shape[0]}')
print(f'  Test-Segmente:      {X_test.shape[0]}')
print(f'  Segmentlänge:       {X_train.shape[1]} Samples')
print(f'  Klassen:            {CLASS_NAMES}')

---
## 2. Datenexploration 🔍

### 2.1 Vibrationssignale im Zeitbereich

Jede Klasse zeigt ein unterschiedliches **Vibrationsmuster**:
- **Normal:** Gleichmäßiges Rauschen, niedrige Amplitude
- **Innenring (IR):** Periodische Impulse mit hoher Repetitionsrate
- **Außenring (OR):** Stärkere periodische Impulse, andere Frequenz
- **Kugel (B):** Unregelmäßige Impulse, variable Amplitude

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Vibrationssignale — Beispiele pro Fehlerklasse', fontsize=16, fontweight='bold')

# Zeitachse in Millisekunden (bei 12 kHz Abtastrate)
fs = 12000
t_ms = np.arange(1024) / fs * 1000

for label_idx, (ax, name, color) in enumerate(
    zip(axes.flat, CLASS_NAMES, COLORS)
):
    # Finde ein Beispiel-Segment dieser Klasse
    mask = y_train == label_idx
    example = X_train[mask][0]
    
    ax.plot(t_ms, example, color=color, linewidth=0.5, alpha=0.8)
    ax.set_title(f'{name}', fontsize=14, fontweight='bold', color=color)
    ax.set_xlabel('Zeit (ms)')
    ax.set_ylabel('Amplitude')
    ax.set_xlim([0, t_ms[-1]])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.2 Frequenzspektrum (FFT)

Die **Fast Fourier Transformation (FFT)** zeigt uns, welche Frequenzen im Signal enthalten sind.
Verschiedene Fehlertypen erzeugen charakteristische **Frequenzmuster**:

- Defekte am **Innenring** erzeugen Impulse bei der Ball Pass Frequency Inner (BPFI)
- Defekte am **Außenring** erzeugen Impulse bei der Ball Pass Frequency Outer (BPFO)
- **Kugelfehler** zeigen Frequenzen bei der Ball Spin Frequency (BSF)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Frequenzspektrum (FFT) — Vergleich der Fehlerklassen', fontsize=16, fontweight='bold')

for label_idx, (ax, name, color) in enumerate(
    zip(axes.flat, CLASS_NAMES, COLORS)
):
    mask = y_train == label_idx
    example = X_train[mask][0]
    
    freqs, magnitude = _compute_spectrum(example, fs=12000)
    
    ax.plot(freqs, magnitude, color=color, linewidth=0.8, alpha=0.8)
    ax.fill_between(freqs, magnitude, alpha=0.2, color=color)
    ax.set_title(f'{name}', fontsize=14, fontweight='bold', color=color)
    ax.set_xlabel('Frequenz (Hz)')
    ax.set_ylabel('Amplitude')
    ax.set_xlim([0, 6000])  # Bis Nyquist-Frequenz
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Klassenverteilung

Ein Blick auf die Verteilung der Klassen im Trainings- und Testdatensatz:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_data, title in [
    (axes[0], y_train, 'Trainingsset'),
    (axes[1], y_test, 'Testset'),
]:
    counts = [np.sum(y_data == i) for i in range(4)]
    bars = ax.bar(CLASS_NAMES, counts, color=COLORS, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Anzahl Segmente')
    
    # Zahlen über den Balken anzeigen
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3. Feature-Extraktion 🔧

### Der klassische Ansatz: Handgemachte Merkmale

Für klassische ML-Algorithmen müssen wir aus jedem Rohsignal-Segment
**aussagekräftige Zahlenwerte (Features)** berechnen.

Wir berechnen **12 Features** pro Segment — aufgeteilt in:

#### Zeitbereich-Features (8):
| Feature | Formel | Was es misst |
|---------|--------|--------------|
| **RMS** | $x_{\text{RMS}} = \sqrt{\frac{1}{N}\sum x_i^2}$ | Signalenergie |
| **Standardabweichung** | $\sigma$ | Streuung |
| **Kurtosis** | 4. Moment | Spitzigkeit (Impulse!) |
| **Schiefe** | 3. Moment | Asymmetrie |
| **Peak-to-Peak** | $x_{\max} - x_{\min}$ | Schwingungsbreite |
| **Scheitelfaktor** | $\frac{|x|_{\max}}{x_{\text{RMS}}}$ | Impulsverhältnis |
| **Formfaktor** | $\frac{x_{\text{RMS}}}{\overline{|x|}}$ | Wellenform |
| **Impulsfaktor** | $\frac{|x|_{\max}}{\overline{|x|}}$ | Impuls-Sensitivität |

#### Frequenzbereich-Features (4):
| Feature | Was es misst |
|---------|--------------|
| **Spektraler Schwerpunkt** | Mittlere Frequenz (gewichtet) |
| **Spektrale Bandbreite** | Frequenz-Streuung |
| **Dominante Frequenz** | Frequenz mit max. Amplitude |
| **Mittlere Frequenz** | Leistungs-gewichtete Frequenz |

In [ ]:
%%time
# Features für Trainings- und Testdaten berechnen
print('Feature-Extraktion für Trainingsdaten...')
X_train_features = extract_all_features(X_train)
X_train_features['Label'] = y_train
X_train_features['Klasse'] = [CLASS_NAMES[l] for l in y_train]

print('\nFeature-Extraktion für Testdaten...')
X_test_features = extract_all_features(X_test)
X_test_features['Label'] = y_test
X_test_features['Klasse'] = [CLASS_NAMES[l] for l in y_test]

# Erste Zeilen anzeigen
print('\n📊 Feature-Tabelle (erste 5 Zeilen pro Klasse):')
display(X_train_features.groupby('Klasse').head(2))

### 3.1 Feature-Verteilungen pro Klasse

Schauen wir uns an, wie gut die Features die einzelnen Klassen trennen:

In [ ]:
# Die 4 aussagekräftigsten Features visualisieren
top_features = ['RMS', 'Kurtosis', 'Scheitelfaktor', 'Spektraler Schwerpunkt']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Feature-Verteilungen pro Fehlerklasse', fontsize=16, fontweight='bold')

for ax, feature in zip(axes, top_features):
    for label_idx, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
        mask = X_train_features['Label'] == label_idx
        values = X_train_features.loc[mask, feature]
        ax.hist(values, bins=30, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feature, fontweight='bold')
    ax.set_xlabel('Wert')
    ax.set_ylabel('Dichte')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### 3.2 Dimensionsreduktion: PCA-Projektion

Die **Principal Component Analysis (PCA)** projiziert unsere 12 Features auf 2 Dimensionen,
sodass wir die Daten in einem 2D-Streudiagramm betrachten können.

**Gut trennbare Cluster = Features sind aussagekräftig!**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Features standardisieren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_features[FEATURE_NAMES])

# PCA auf 2 Dimensionen
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
for label_idx, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
    mask = y_train == label_idx
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=name, alpha=0.5, s=20, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Varianz)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Varianz)', fontsize=12)
ax.set_title('PCA-Projektion der extrahierten Features', fontsize=16, fontweight='bold')
ax.legend(fontsize=12, markerscale=3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Erklärte Varianz: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%, '
      f'Gesamt={sum(pca.explained_variance_ratio_)*100:.1f}%')

---
## 4. Klassisches ML: Random Forest 🌲

### Wie funktioniert ein Random Forest?

1. **Viele Entscheidungsbäume** werden auf zufälligen Teilmengen der Daten trainiert
2. Jeder Baum "stimmt ab" → die **Mehrheitsentscheidung** gewinnt
3. Vorteile: Robust, gute Genauigkeit, berechnet **Feature Importances**

### Pipeline:
```
Rohsignal → Segmentierung → Feature-Extraktion (12 Features) → StandardScaler → Random Forest → Vorhersage
```

**Wichtig:** Der Random Forest arbeitet mit den **handgemachten Features**, nicht mit den Rohsignalen!

In [ ]:
# Random Forest trainieren
print('='*60)
print('TRAINING: Random Forest Classifier')
print('='*60)

rf_result = train_classical(
    X_train_features[FEATURE_NAMES],
    y_train,
    n_estimators=100,  # 100 Entscheidungsbäume
)
rf_model = rf_result['model']

# Evaluation auf Testdaten
print('\n' + '='*60)
print('EVALUATION: Random Forest auf Testdaten')
print('='*60)

rf_eval = evaluate_classical(
    rf_model,
    X_test_features[FEATURE_NAMES],
    y_test,
    class_names=CLASS_NAMES,
)

### 4.1 Konfusionsmatrix

Die Konfusionsmatrix zeigt, welche Klassen gut erkannt werden und wo Verwechslungen auftreten:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Konfusionsmatrix
ax = axes[0]
cm = rf_eval['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=1, linecolor='white')
ax.set_xlabel('Vorhersage', fontsize=12)
ax.set_ylabel('Wahrheit', fontsize=12)
ax.set_title(f'Random Forest — Konfusionsmatrix\n(Accuracy: {rf_eval["accuracy"]*100:.1f}%)',
             fontsize=14, fontweight='bold')

# Feature Importances 
ax = axes[1]
importances = rf_eval['feature_importances']
sorted_idx = np.argsort(importances)
ax.barh(range(len(importances)), importances[sorted_idx], color='#2ecc71')
ax.set_yticks(range(len(importances)))
ax.set_yticklabels(np.array(FEATURE_NAMES)[sorted_idx])
ax.set_xlabel('Wichtigkeit', fontsize=12)
ax.set_title('Feature Importances (Random Forest)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n🎯 Random Forest Ergebnis:')
print(f'   Accuracy:  {rf_eval["accuracy"]*100:.2f}%')
print(f'   F1-Score:  {rf_eval["f1_macro"]*100:.2f}%')
print(f'   Trainingszeit: {rf_result["train_time"]:.2f}s')

---
## 5. Deep Learning: 1D-CNN 🧠

### Was ist anders beim Deep Learning?

| | Klassisches ML | Deep Learning |
|---|---|---|
| **Input** | Handgemachte Features | **Rohsignal** direkt |
| **Feature-Extraktion** | Manuell (Domain-Wissen nötig) | **Automatisch** gelernt |
| **Modell** | Random Forest (Bäume) | 1D-CNN (Faltungsschichten) |
| **Interpretierbarkeit** | Feature Importances | Schwieriger ("Black Box") |

### Unser CNN-Aufbau:

```
Rohsignal (1024 Punkte)
    │
    ├─ Conv1D(32, kernel=7)  → Erkennt grobe Muster
    ├─ Conv1D(64, kernel=5)  → Erkennt feinere Muster  
    ├─ Conv1D(128, kernel=3) → Erkennt komplexe Muster
    │
    ├─ GlobalAveragePooling  → Zusammenfassung
    ├─ Dense(64) + Dropout   → Entscheidungsschicht
    └─ Dense(4, Softmax)     → 4 Klassen-Wahrscheinlichkeiten
```

**Kernidee:** Die Conv1D-Filter funktionieren wie automatisch gelernte Feature-Extraktoren!

In [ ]:
# CNN erstellen
cnn = build_cnn(input_length=1024, num_classes=4)

# Modell-Zusammenfassung anzeigen
print('='*60)
print('CNN-ARCHITEKTUR')
print('='*60)
cnn.summary()

In [ ]:
# CNN trainieren
print('='*60)
print('TRAINING: 1D Convolutional Neural Network')
print('='*60)

cnn_result = train_cnn(
    cnn, X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.15,
    verbose=1,
)

### 5.1 Trainingsverlauf

Die Loss- und Accuracy-Kurven zeigen, wie das CNN über die Epochen lernt:

In [ ]:
history = cnn_result['history'].history

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss-Kurve
ax = axes[0]
ax.plot(history['loss'], 'b-', linewidth=2, label='Training')
ax.plot(history['val_loss'], 'r--', linewidth=2, label='Validierung')
ax.set_xlabel('Epoche', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Verlust (Loss) über die Epochen', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Accuracy-Kurve
ax = axes[1]
ax.plot(np.array(history['accuracy'])*100, 'b-', linewidth=2, label='Training')
ax.plot(np.array(history['val_accuracy'])*100, 'r--', linewidth=2, label='Validierung')
ax.set_xlabel('Epoche', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Genauigkeit (Accuracy) über die Epochen', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CNN auf Testdaten evaluieren
print('='*60)
print('EVALUATION: 1D-CNN auf Testdaten')
print('='*60)

cnn_eval = evaluate_cnn(cnn, X_test, y_test, class_names=CLASS_NAMES)

In [ ]:
# CNN Konfusionsmatrix
fig, ax = plt.subplots(figsize=(8, 6))

cm_cnn = cnn_eval['confusion_matrix']
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=1, linecolor='white')
ax.set_xlabel('Vorhersage', fontsize=12)
ax.set_ylabel('Wahrheit', fontsize=12)
ax.set_title(f'1D-CNN — Konfusionsmatrix\n(Accuracy: {cnn_eval["accuracy"]*100:.1f}%)',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. Vergleich: Klassisches ML vs. Deep Learning ⚖️

Jetzt vergleichen wir die beiden Ansätze systematisch:

In [ ]:
# Vergleichstabelle erstellen
comparison = pd.DataFrame({
    'Metrik': [
        'Test-Accuracy (%)',
        'F1-Score (%)',
        'Trainingszeit (s)',
        'Feature-Engineering nötig?',
        'Interpretierbarkeit',
        'Modellkomplexität',
    ],
    'Random Forest': [
        f'{rf_eval["accuracy"]*100:.2f}',
        f'{rf_eval["f1_macro"]*100:.2f}',
        f'{rf_result["train_time"]:.2f}',
        'Ja (12 Features)',
        'Hoch (Feature Importances)',
        'Niedrig',
    ],
    '1D-CNN': [
        f'{cnn_eval["accuracy"]*100:.2f}',
        f'{cnn_eval["f1_macro"]*100:.2f}',
        f'{cnn_result["train_time"]:.1f}',
        'Nein (Rohsignal)',
        'Niedrig (Black Box)',
        'Hoch',
    ],
})

print('\n📊 VERGLEICH: Random Forest vs. 1D-CNN')
print('='*60)
display(comparison.style.set_properties(**{'text-align': 'center'})
        .hide(axis='index'))

In [ ]:
# Visueller Vergleich: Konfusionsmatrizen nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, cm, title, cmap, eval_result in [
    (axes[0], rf_eval['confusion_matrix'], 'Random Forest', 'Greens', rf_eval),
    (axes[1], cnn_eval['confusion_matrix'], '1D-CNN', 'Blues', cnn_eval),
]:
    # Normalisierte Konfusionsmatrix (Prozent)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap=cmap, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=1, linecolor='white', vmin=0, vmax=100)
    ax.set_xlabel('Vorhersage', fontsize=12)
    ax.set_ylabel('Wahrheit', fontsize=12)
    ax.set_title(f'{title}\nAccuracy: {eval_result["accuracy"]*100:.1f}% | '
                 f'F1: {eval_result["f1_macro"]*100:.1f}%',
                 fontsize=14, fontweight='bold')

plt.suptitle('Vergleich der Konfusionsmatrizen (in %)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy-Vergleich als Balkendiagramm
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Random Forest\n(mit Features)', '1D-CNN\n(Rohsignal)']
accuracies = [rf_eval['accuracy']*100, cnn_eval['accuracy']*100]
f1_scores = [rf_eval['f1_macro']*100, cnn_eval['f1_macro']*100]

x = np.arange(len(methods))
width = 0.3

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1-Score', color='#2ecc71', edgecolor='white')

# Werte über den Balken
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.3,
                f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylim([0, 105])
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=13)
ax.set_ylabel('Prozent (%)', fontsize=12)
ax.set_title('Vergleich: Klassisches ML vs. Deep Learning', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Fazit & Zusammenfassung 📝

### Was haben wir gelernt?

1. **Vibrationssignale** von Wälzlagern enthalten charakteristische Muster für verschiedene Fehlertypen

2. **Klassisches ML (Random Forest):**
   - Benötigt **manuelle Feature-Extraktion** (Domain-Wissen erforderlich)
   - Gut interpretierbar dank **Feature Importances**
   - Schnelles Training, wenig Rechenleistung nötig
   - Sehr gute Ergebnisse bei gut gewählten Features

3. **Deep Learning (1D-CNN):**
   - Arbeitet direkt mit **Rohsignalen** — keine manuelle Feature-Extraktion nötig!
   - Lernt automatisch die relevanten Merkmale ("end-to-end learning")
   - Kann subtilere Muster erkennen als handgemachte Features
   - Benötigt mehr Daten und Rechenleistung
   - Weniger interpretierbar ("Black Box")

### Wann welchen Ansatz wählen?

| Situation | Empfehlung |
|-----------|------------|
| Wenig Daten, klares Domain-Wissen | **Klassisches ML** |
| Viele Daten, komplexe Muster | **Deep Learning** |
| Interpretierbarkeit wichtig (z.B. Zertifizierung) | **Klassisches ML** |
| Schnelle Entwicklung ohne Expertenwissen | **Deep Learning** |
| Echtzeit-Anforderung auf Edge-Geräten | **Klassisches ML** (schnellere Inferenz) |

### Ausblick

- **Transfer Learning:** Vortrainierte Modelle auf neue Maschinentypen übertragen
- **Anomalieerkennung:** Nur "Normal" trainieren, Abweichungen erkennen
- **Explainable AI (XAI):** Deep Learning interpretierbar machen (Grad-CAM, SHAP)
- **Edge Deployment:** Modelle auf Mikrocontrollern für Echtzeit-Überwachung

---
*Erstellt für die Probevorlesung "Mustererkennung in einem Produktionsprozess"*

In [ ]:
# Optional: Modelle speichern
import joblib

MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Random Forest speichern
joblib.dump(rf_model, os.path.join(MODELS_DIR, 'random_forest.pkl'))
print('✓ Random Forest gespeichert')

# CNN speichern
cnn.save(os.path.join(MODELS_DIR, 'cnn_model.keras'))
print('✓ CNN gespeichert')

print(f'\nModelle gespeichert in: {MODELS_DIR}')